In [ ]:
import os
from copy import deepcopy
import numpy as np
import random
import warnings

import torch
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from utils import calc_logit_norm, validate, extract_features, \
    visualize_features, get_reduced_features, add_gaussian_noise, \
    entropy_loss, collect_params, configure_model, adapt_model, \
    get_animated_features, get_partial_dataloader, animate_3d_features, logit_adjusted_adaptation,\
    class_counts

from models import CNN
os.environ["CUDA_VISIBLE_DEVICES"] = "2"  # Replace "0" with the desired GPU device index


In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning)

random_seed = 2025

torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
np.random.seed(random_seed)
random.seed(random_seed)

In [ ]:
# Parameters
LOG_FREQUENCY =  1000
DO_ADJUST_LOGITS = False
TAU = 1.0
PARTIAL_CLASSES = [i for i in range(1)]

In [ ]:
test_class_distribution = []
for i in range(10):
    if i in PARTIAL_CLASSES:
        test_class_distribution.append(1.0/len(PARTIAL_CLASSES))
    else:
        test_class_distribution.append(0.0)


transform = transforms.Compose([
    transforms.ToTensor(),  # converts to tensor and scales image pixel values to [0, 1]
    transforms.Normalize((0.1307,), (0.3081,))  # normalize using MNIST's mean and std
])

train_dataset = datasets.MNIST(root='/home/thilina/SSD2/thilina/datasets/mnist', train=True, download=False, transform=transform)
test_dataset  = datasets.MNIST(root='/home/thilina/SSD2/thilina/datasets/mnist', train=False, download=False, transform=transform)

model = torch.load('/home/thilina/SSD2/thilina/test-time-adaptation-further_experiments/classification/simple_classifier/mnist_model.pth')
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Noisy data
noisy_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: add_gaussian_noise(x, severity=5)),
    transforms.Normalize((0.1307,), (0.3081,))
])

noisy_test_dataset = datasets.MNIST(root='/home/thilina/SSD2/thilina/datasets/mnist', train=False, download=False, transform=noisy_transform)

# when doing TTA, dataloader should be shuffled
noisy_train_loader = DataLoader(noisy_test_dataset, batch_size=64, shuffle=True)

noisy_test_loader = DataLoader(noisy_test_dataset, batch_size=64, shuffle=False)


########################################################################
# Adapt the model to balanced dataset
# test_acc_list, reduced_feature_list, _ = adapt_model(model, noisy_train_loader, noisy_test_loader, reducer, log_frequency=500, feature_layer='fc1')
# print(f"Class Counts: {class_counts(noisy_train_loader)}")
balanced_test_acc_list, _, _, balanced_gradient_norm_dict = logit_adjusted_adaptation(model, noisy_train_loader, noisy_test_loader, do_adjust_logits=False,
                                                                                      log_frequency=LOG_FREQUENCY, epochs=40)

# get_animated_features(reduced_feature_list, initial_labels, "balanced_adaptation_2d.mp4")
# animate_3d_features(reduced_feature_list, initial_labels, "balanced_adaptation_umap_3d.mp4")

########################################################################
# Adapt the model to Imbalanced dataset
imbalanced_test_loader = get_partial_dataloader(noisy_test_dataset, PARTIAL_CLASSES, final_samples=10000)
print(f"Class Counts: {class_counts(imbalanced_test_loader)}")

model = torch.load('/home/thilina/SSD2/thilina/test-time-adaptation-further_experiments/classification/simple_classifier/mnist_model.pth')

test_acc_list, _, _, gradient_norm_dict = logit_adjusted_adaptation(model, imbalanced_test_loader, noisy_test_loader, class_distribution=test_class_distribution,
                                                                    do_adjust_logits=False, log_frequency=LOG_FREQUENCY, tau=TAU, epochs=40)


In [ ]:
model = torch.load('/home/thilina/SSD2/thilina/test-time-adaptation-further_experiments/classification/simple_classifier/mnist_model.pth')

adjusted_test_acc_list, reduced_feature_list, _, la_gradient_norm_dict = logit_adjusted_adaptation(model, imbalanced_test_loader, noisy_test_loader, class_distribution=test_class_distribution,
                                                                    do_adjust_logits=True, log_frequency=LOG_FREQUENCY, tau=TAU, epochs=40)



In [ ]:
import matplotlib.pyplot as plt

digits = [8]

# Gradient Norms for balanced adaptation
fig, ax = plt.subplots()
steps = np.arange(len(balanced_gradient_norm_dict[0]))

for ii in digits:
    ax.plot(steps, balanced_gradient_norm_dict[ii], label=f"Class {ii}")

ax.set(xlabel='step', ylabel='Gradient Norm',
       title='Balanced Adaptation')
ax.legend()
fig.show()

# Gradient Norms for imbalanced adaptation
fig, ax = plt.subplots()
steps = np.arange(len(gradient_norm_dict[0]))

for ii in digits:
    ax.plot(steps, gradient_norm_dict[ii], label=f"Class {ii}")

ax.set(xlabel='step', ylabel='Gradient Norm',
       title='Imbalanced Adaptation')
ax.legend()
fig.show()

# Gradient Norms for imbalanced adaptation with Logit Adjustment
fig, ax = plt.subplots()
steps = np.arange(len(la_gradient_norm_dict[0]))

for ii in digits:
    ax.plot(steps, la_gradient_norm_dict[ii], label=f"Class {ii}")

ax.set(xlabel='step', ylabel='Gradient Norm',
       title='Imbalanced Adaptation with Logit Adjustment')
ax.legend()
fig.show()

In [ ]:
# plot adjusted and non-adjusted test accuracies
import matplotlib.pyplot as plt

plt.figure(figsize=(6,3))
step_list = [x[1] for x in test_acc_list]
accuracy_list = [x[0] for x in test_acc_list]
plt.plot(step_list, accuracy_list)
adjusted_accuracy_list = [x[0] for x in adjusted_test_acc_list]
plt.plot(step_list, adjusted_accuracy_list)

plt.legend(['Imbalanced', 'Logit Adjusted Imbalanced'])

plt.title(f"Class Imbalance {len(PARTIAL_CLASSES)}/10")
plt.xlabel("Steps")
plt.ylabel("Test Accuracy")
plt.ylim([0.0, 1.0])

In [ ]:
 
# tau = +1.0
# class imbalance 1/10 - Digit 0 only
# original_logits = [-0.5057742595672607, -2.440204620361328, 4.544600486755371, 0.9368734359741211, -9.204654693603516, -0.506374180316925, -10.393024444580078, 0.7086231708526611, -11.251418113708496, -11.69865894317627]
# adjusted_logits = [-0.5057742595672607, -30.071226119995117, -23.086421966552734, -26.694149017333984, -36.83567810058594, -28.13739585876465, -38.0240478515625, -26.92239761352539, -38.88243865966797, -39.329681396484375]

# class imbalance 3/10 - Digit 0, 1, 2
original_logits = [-10.398523330688477, 7.32331657409668, 2.2901735305786133, -0.4406801462173462, -5.250664234161377, -0.9346347451210022, -13.246556282043457, 3.547624111175537, -12.570435523986816, -12.74984073638916]
adjusted_logits = [-11.497135162353516, 6.224704265594482, 1.191561222076416, -28.071701049804688, -32.88168716430664, -28.565656661987305, -40.87757873535156, -24.083396911621094, -40.20145797729492, -40.380863189697266]

# tau = -1.0
# class imbalance 1/10 - Digit 0 only
# original_logits  =  [-0.5057742595672607, -2.440204620361328, 4.544600486755371, 0.9368734359741211, -9.204654693603516, -0.506374180316925, -10.393024444580078, 0.7086231708526611, -11.251418113708496, -11.69865894317627]
# adjusted_logits = [-0.5057742595672607, 25.19081687927246, 32.175621032714844, 28.567893981933594, 18.426366806030273, 27.12464714050293, 17.23799705505371, 28.339645385742188, 16.37960433959961, 15.93236255645752]



In [ ]:
# Create the plot
fig, ax = plt.subplots()
indices = np.arange(len(original_logits))
bar_width = 0.35
# Plot bars for each array with an offset so they appear side by side
bars1 = ax.bar(indices - bar_width/2, original_logits, bar_width, label='Logits')
bars2 = ax.bar(indices + bar_width/2, adjusted_logits, bar_width, label='Adjusted Logits')
plt.xlabel('Index')
plt.ylabel('Logit Value')
plt.title('Logits vs Adjusted Logits')
ax.legend()
plt.show()

In [ ]:

original_logits = torch.Tensor(original_logits).view(1,10)
adjusted_logits = torch.Tensor(adjusted_logits).view(1,10)

original_softmax = torch.nn.functional.softmax(original_logits, dim=1)
adjusted_softmax = torch.nn.functional.softmax(adjusted_logits, dim=1)

original_softmax, adjusted_softmax

In [ ]:
from utils import entropy_loss

original_entropy = entropy_loss(original_softmax)
adjusted_entropy = entropy_loss(adjusted_softmax)

original_entropy, adjusted_entropy